# evaluate_checkpoints

Run the trials again on checkpoints a run already saved: **no training**. The base model loads once; each step
applies that step's saved LoRA adapter and goes through the same evaluation a training run gives its checkpoints:

1. the decision trials (`model_estimating.estimation_trials_per_persona`, 1000 for the one-persona runs), answered by
   the model,
2. the rule recovered from those answers,
3. the report questions, scored against the recovered rule and the true one.

Step `0` is the untrained base model. Everything comes from the run's own `results/<run>/settings.json`, so the
answers should match the run's: the **Reproduction** cell checks that, decision for decision.

| notebook | asks the decision trials? | asks report questions? | trains? |
|---|---|---|---|
| `run_experiment.ipynb` | yes | yes | yes |
| **`evaluate_checkpoints.ipynb`** (this one) | **yes** | **yes** | no |
| `ask_only.ipynb` | no (refits the rule from the run's saved decisions) | yes | no |

Needs, on this machine: `checkpoints/<run>/decision/step-<N>/` for each step and `results/<run>/settings.json`.
`download_checkpoint.ipynb` fetches both from the Hub (paste the run's `.../tree/main/checkpoints/<run>/decision`
link, or one step's folder).

Writes only into `results/<run>/evaluate_only/<NAME>/`: the files a training run writes per checkpoint
(`selections_`, `recovered_`, `weight_reports_`, `reports_raw_`, `persona_scores_`, `learned_vs_reported_`, ...), a
`results.csv` of these steps alone, `settings.json` (what was asked) and `timing.csv`. Never appends to
`results/results.csv` and never changes a file of the run.

Edit the settings cell only. An 8B run needs a GPU: about the evaluation time of one checkpoint of the run per step.

## Settings

Edit this cell only, then run the cells below it.

In [ ]:
RUN = "20261116-221120"     # a run id, or its timestamp prefix (results/<run>/ must hold settings.json)
STEPS = [0, 1500, 3000]     # saved decision steps; 0 = the base model, no adapter
NAME = "recheck"            # output folder: results/<run>/evaluate_only/<NAME>/

# Everything below None / {} keeps the run's own settings, which is what the reproduction check expects.
QUESTIONS_FROM = None       # a settings file to take report_schema (the question set) from
ONLY = None                 # a list of question names, to ask only those
SHOW_NEITHER = None         # True / False: show option C in report trials
REPLY_PREFIX = None         # a string the reply starts with ("" = none)
OVERRIDES = {}              # {"block.field": value}, e.g. for a smaller card or a quicker pass:
                            #   {"compute.decision_eval_batch_size": 16, "compute.eval_batch_size": 16}
                            #   {"model_estimating.estimation_trials_per_persona": 200}
                            # On a CPU (slow; 0.6B smoke runs only):
                            #   {"compute.device": "cpu", "compute.mixed_precision": "no", "model_hyperparameters.quantization": "none"}

## Set up

In [ ]:
import os, sys, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "evaluate_saved.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("run this notebook from inside the project folder")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)

from src import ask_only as A
from src import evaluate_saved as ES
print("project:", ROOT)

## Which adapters are saved

In [ ]:
run_dir = A.find_run(RUN)
saved = A.available_steps(run_dir.name)
print(run_dir.name)
print("saved steps here:", saved)
missing = [s for s in STEPS if s and s not in saved]
if missing:
    raise FileNotFoundError(f"steps {missing} have no adapter here: fetch them with download_checkpoint.ipynb "
                            f"(https://huggingface.co/adamchanna/bdp-plunkett-qwen3/tree/main/checkpoints/{run_dir.name}/decision)")

## Evaluate

In [ ]:
LOADED = globals().get("LOADED")    # the base model, kept across runs of this cell (same model only)
t = time.time()
out_dir, timing, LOADED = ES.evaluate(RUN, STEPS, NAME, QUESTIONS_FROM, ONLY, SHOW_NEITHER, REPLY_PREFIX, OVERRIDES,
                                      loaded=LOADED)
print(out_dir, f"{time.time() - t:.0f}s")
pd.DataFrame(timing)

## Reproduction

In [ ]:
# The share of decisions equal to the run's own at each step. With the run's settings on the same kind of card this
# should be 1.0; another card, batch size or quantization can move a probability in the last digits and flip a
# near-tie, so a rate just under 1 with a small max_probability_gap is the hardware, not a different model.
pd.DataFrame([r for r in (ES.reproduction(run_dir, out_dir, s) for s in STEPS) if r])

## Results

In [ ]:
rows = pd.read_csv(out_dir / "results.csv")
cols = [c for c in ["stage", "checkpoint_step", "block", "question", "decision_accuracy", "parse_rate",
                    "faithfulness", "faithfulness_shuffled", "reported_vs_hidden"] if c in rows.columns]
rows[cols]

In [ ]:
# The persona readouts and the trial regression on what was just asked, as for a training run's folder.
import subprocess
for tool in (["tools/persona_readouts.py", str(out_dir), "--boot", "2000"], ["tools/trial_regression.py", str(out_dir)]):
    subprocess.run([sys.executable, *tool], check=False)